# Evaluate a decider model

A decision model answers a typed question with a probability distribution instead of free text. This
notebook scores one against gold labels on 100 support tickets, offline, in a few minutes:

1. load the questions, tickets and stored responses;
2. build the eval model from the questions;
3. run `BulkStructuredModelEvaluator`;
4. read accuracy, coverage, per-class metrics, the confusion matrix and calibration;
5. measure accept-or-escalate at a target accuracy;
6. put confidence intervals on the numbers.

An optional last section runs Strands Decider live. The guide that goes with this notebook is
[Evaluating Decision Models](https://awslabs.github.io/stickler/Guides/Evaluation/decision-models/).

> ## Warning: the stored responses are synthetic placeholders
>
> `fixtures/systemone/responses.jsonl` was **not produced by any model**. A seeded script drew it from
> the gold labels so this notebook runs offline. Every score below is computed from those placeholders.
> **Do not cite any number in this notebook as a measurement of Strands Decider or any other model.**
>
> The provenance cell below says whether that is still true. Once the fixture is re-recorded with
> `record_systemone.py`, `synthetic` reads `false` and the numbers are a real measurement.

In [1]:
import inspect
import json
from pathlib import Path

import stickler

print(f"stickler {stickler.__version__} from {inspect.getfile(stickler)}")

FIXTURE = Path("fixtures/systemone")


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


questions = json.loads((FIXTURE / "questions.json").read_text())
tickets = read_jsonl(FIXTURE / "tickets.jsonl")
responses = {r["id"]: r["response"] for r in read_jsonl(FIXTURE / "responses.jsonl")}
provenance = json.loads((FIXTURE / "provenance.json").read_text())

print(f"{len(questions)} questions, {len(tickets)} tickets, {len(responses)} responses\n")
print(json.dumps(provenance, indent=2))
SYNTHETIC = bool(provenance.get("synthetic"))


def caveat():
    # Printed under every score while the stored responses are synthetic placeholders.
    if SYNTHETIC:
        print("\n[synthetic placeholder responses: not a measurement of any model; do not cite]")


caveat()

stickler 1.0.1 from /private/tmp/stickler-469/src/stickler/__init__.py
6 questions, 100 tickets, 100 responses

{
  "model_id": "StrandsAgents/strands-decider-2B-qwen3.5-v1-2610",
  "response_model": "synthetic-placeholder",
  "hand_authored": true,
  "synthetic": true,
  "recorded": null,
  "strands_decider_version": null,
  "generator": {
    "script": "make_synthetic_responses.py",
    "seed": 469
  },
  "note": "NOT a recording. responses.jsonl is synthetic, drawn by make_synthetic_responses.py from the gold labels with a seeded generator, so the notebook and tests run offline. No model produced these answers. model_id names the model whose output they stand in for. Do not cite any score computed from them as a measurement. Replace them with record_systemone.py."
}

[synthetic placeholder responses: not a measurement of any model; do not cite]


### What's in the files

A quick look at each file loaded above, so you don't have to open them.

`questions.json`: the System One `questions` dict, in full. This is exactly what gets sent to the model.

In [2]:
print(json.dumps(questions, indent=2))

{
  "queue": {
    "type": "choice",
    "instructions": "Which support team should handle this ticket?",
    "criteria": {
      "billing": "Charges, invoices, payment methods, refunds of money already paid, plan prices",
      "technical": "The product misbehaving: errors, crashes, outages, integrations, data not syncing",
      "account": "Logging in, passwords, two-factor, user seats, account ownership, closing an account",
      "shipping": "Physical deliveries: tracking, delays, lost or damaged parcels, addresses, returns in transit",
      "sales": "Buying more: new plans, upgrades, quotes, volume pricing, trials, demos"
    }
  },
  "priority": {
    "type": "choice",
    "instructions": "What priority should this ticket get in the support queue?",
    "criteria": {
      "low": "No harm in waiting a few days: general questions, suggestions, feedback",
      "normal": "Should be handled within a business day; the customer is affected but not blocked",
      "high": "The custome

`tickets.jsonl`: one ticket per line, with its text (`state`), the questions asked for it, and the gold label for each. The first three tickets:

In [3]:
for t in tickets[:3]:
    print(json.dumps(t, indent=2))
print(f"... {len(tickets)} tickets in total")

{
  "id": "t001",
  "state": "Hi, I was charged twice for my March subscription. Both charges show $49 on my card statement. Can you refund the duplicate?",
  "asked": [
    "queue",
    "priority",
    "ticket_type",
    "frustration",
    "needs_refund",
    "urgent"
  ],
  "gold": {
    "queue": "billing",
    "priority": "normal",
    "ticket_type": "refund_request",
    "frustration": 1,
    "needs_refund": true,
    "urgent": false
  }
}
{
  "id": "t002",
  "state": "The dashboard has been showing a 502 error for the last hour and none of our team can log any orders. We run a restaurant and dinner service starts at 5.",
  "asked": [
    "queue",
    "priority",
    "ticket_type",
    "frustration",
    "needs_refund",
    "urgent"
  ],
  "gold": {
    "queue": "technical",
    "priority": "high",
    "ticket_type": "bug",
    "frustration": 2,
    "needs_refund": false,
    "urgent": true
  }
}
{
  "id": "t003",
  "state": "How do I add a second admin to our workspace? I couldn't

`responses.jsonl`: one System One response per ticket, as the model returned it. The first one:

In [4]:
first = next(iter(responses))
print(json.dumps({"id": first, "response": responses[first]}, indent=2))
print(f"... {len(responses)} responses in total")

{
  "id": "t001",
  "response": {
    "model": "synthetic-placeholder",
    "answers": {
      "queue": {
        "type": "choice",
        "choice": "billing",
        "probabilities": {
          "billing": 0.7846,
          "technical": 0.0163,
          "account": 0.0387,
          "shipping": 0.0635,
          "sales": 0.0969
        },
        "confidence": 0.7308
      },
      "priority": {
        "type": "choice",
        "choice": "normal",
        "probabilities": {
          "low": 0.208,
          "normal": 0.7149,
          "high": 0.0771
        },
        "confidence": 0.5723
      },
      "ticket_type": {
        "type": "choice",
        "choice": "complaint",
        "probabilities": {
          "question": 0.0733,
          "bug": 0.0068,
          "refund_request": 0.2676,
          "complaint": 0.6457,
          "feature_request": 0.0066
        },
        "confidence": 0.5571
      },
      "frustration": {
        "type": "score",
        "score": 2.25,
      

## 1. The questions are the schema

`questions.json` is the `questions` dict sent to the model, exactly as sent. `model_from_questions` turns
it into a Stickler model with one decision field per question: Choice for one option out of a set, Score
for a level on an ordered scale, Noul for yes or no. `weights` is the only Stickler-specific input; here
routing (`queue`) counts double in each ticket's `overall_score`.

In [5]:
from stickler.decisions import decision_fields
from stickler.integrations.systemone import from_systemone, model_from_questions

Ticket = model_from_questions(questions, weights={"queue": 2.0}, model_name="Ticket")

for name, spec in decision_fields(Ticket).items():
    labels = [label.split(":")[0] for label in spec.labels]  # Score levels are long; show their names
    print(f"{name:13} {spec.kind:7} {labels}")

queue         choice  ['billing', 'technical', 'account', 'shipping', 'sales']
priority      choice  ['low', 'normal', 'high']
ticket_type   choice  ['question', 'bug', 'refund_request', 'complaint', 'feature_request']
frustration   score   ['Calm', 'Annoyed', 'Frustrated', 'Furious']
needs_refund  noul    ['no', 'yes']
urgent        noul    ['no', 'yes']


## 2. One ticket

Gold labels are an option key (Choice), a level number from 0 (Score), or `true`/`false` (Noul). A
`null` gold means the question was not asked for that ticket.

`from_systemone` returns the response's `answers` unchanged. Only `probabilities` (Choice, Score) and
`noul` are read; the answer's own `choice`, `score`, `confidence` and `legend` are ignored. The predicted
answer is the most probable one, and its confidence is that probability.

In [6]:
t = tickets[0]
answers = from_systemone(responses[t["id"]])
pred = Ticket.from_json(answers)
conf = pred.get_all_confidences()

print(t["state"], "\n")
print(f"{'question':13} {'gold':16} {'predicted':16} {'confidence':>10}")
for q in questions:
    print(f"{q:13} {str(t['gold'][q]):16} {str(getattr(pred, q)):16} {conf.get(q, float('nan')):>10.3f}")

print("\nthe raw queue answer:", json.dumps(answers["queue"]))
caveat()

Hi, I was charged twice for my March subscription. Both charges show $49 on my card statement. Can you refund the duplicate? 

question      gold             predicted        confidence
queue         billing          billing               0.785
priority      normal           normal                0.715
ticket_type   refund_request   complaint             0.646
frustration   1                3                     0.468
needs_refund  True             True                  0.930
urgent        False            False                 0.965

the raw queue answer: {"type": "choice", "choice": "billing", "probabilities": {"billing": 0.7846, "technical": 0.0163, "account": 0.0387, "shipping": 0.0635, "sales": 0.0969}, "confidence": 0.7308}

[synthetic placeholder responses: not a measurement of any model; do not cite]


## 3. Run the evaluator

`BulkStructuredModelEvaluator` finds the decision fields in `target_schema` and computes their metrics
with no configuration. `evaluate` is a function so section 5 can reuse it on two halves of the data.

In [7]:
from stickler.structured_object_evaluator.bulk_structured_model_evaluator import (
    BulkStructuredModelEvaluator,
)


def evaluate(ticket_list):
    ev = BulkStructuredModelEvaluator(target_schema=Ticket)
    for t in ticket_list:
        pred = Ticket.from_json(from_systemone(responses[t["id"]]))
        ev.update(Ticket(**t["gold"]), pred, doc_id=t["id"])
    return ev.compute()


res = evaluate(tickets)
fields = res.decision_metrics["fields"]
print(f"documents: {res.decision_metrics['document_count']}, errors: {res.errors}")

documents: 100, errors: []


## 4. What the run says

### Accuracy and coverage

*Asked* excludes tickets whose gold is `null` for that question. *Missing* is an asked question that got
no answer (a refusal, error or timeout): it counts as wrong in `accuracy`, and is left out of
`answered_accuracy` and of every calibration metric.

In [8]:
print(f"{'question':13} {'type':7} {'asked':>5} {'missing':>7} {'accuracy':>8} {'answered':>8}")
for name, f in fields.items():
    c = f["correctness"]
    cov = c["coverage"]
    print(
        f"{name:13} {f['kind']:7} {cov['ok'] + cov['missing']:>5} {cov['missing']:>7} "
        f"{c['accuracy']:>8.3f} {c['answered_accuracy']:>8.3f}"
    )

s = fields["frustration"]["correctness"]
print(f"\nfrustration (Score): qwk {s['qwk']:.3f}, mae_argmax {s['mae_argmax']:.3f}, mae_expected {s['mae_expected']:.3f}")
caveat()

question      type    asked missing accuracy answered
queue         choice     98       1    0.786    0.794
priority      choice    100       1    0.710    0.717
ticket_type   choice     97       1    0.722    0.729
frustration   score      99       1    0.556    0.561
needs_refund  noul       96       0    0.823    0.823
urgent        noul       98       2    0.765    0.781

frustration (Score): qwk 0.635, mae_argmax 0.500, mae_expected 0.479

[synthetic placeholder responses: not a measurement of any model; do not cite]


### Per class for `queue`

Accuracy hides which teams get misrouted tickets. `per_class` gives precision, recall and F1 per option,
and `macro_f1` their unweighted mean.

In [9]:
q = fields["queue"]["correctness"]
print(f"{'option':10} {'precision':>9} {'recall':>7} {'f1':>6} {'support':>8}")
for label, m in q["per_class"].items():
    print(f"{label:10} {m['precision']:>9.3f} {m['recall']:>7.3f} {m['f1']:>6.3f} {m['support']:>8}")
print(f"\nmacro_f1 {q['macro_f1']:.3f}")
caveat()

option     precision  recall     f1  support
billing        0.760   0.760  0.760       25
technical      0.857   0.828  0.842       29
account        0.722   0.929  0.813       14
shipping       0.909   0.667  0.769       15
sales          0.733   0.733  0.733       15

macro_f1 0.783

[synthetic placeholder responses: not a measurement of any model; do not cite]


### Confusion matrix

Rows are gold, columns predicted. `(no answer)` is where missing answers land.

In [10]:
labels = fields["queue"]["labels"]
cols = labels + ["(no answer)"]
print(f"{'gold / pred':12}" + "".join(f"{c:>12}" for c in cols))
for g in labels:
    row = q["confusion"].get(g, {})
    print(f"{g:12}" + "".join(f"{row.get(c, 0):>12}" for c in cols))
caveat()

gold / pred      billing   technical     account    shipping       sales (no answer)
billing               19           0           1           1           4           0
technical              0          24           4           0           0           1
account                0           1          13           0           0           0
shipping               4           1           0          10           0           0
sales                  2           2           0           0          11           0

[synthetic placeholder responses: not a measurement of any model; do not cite]


### Calibration

Can you trust the confidence? `auroc` is how well confidence separates right answers from wrong ones
(1.0 is perfect, 0.5 is chance). `ece_mass` is the gap between confidence and accuracy over equal-mass
bins (0 is perfect). The proper scoring rules are per type: `brier_multiclass` and `nll` for Choice,
`rps` for Score, `brier` and `log_loss` on P(yes) for Noul, which also reports `auroc_label`
(P(yes) against gold). Lower is better for all of them.

In [11]:
extra = {"choice": ("brier_multiclass", "nll"), "score": ("rps",), "noul": ("brier", "log_loss", "auroc_label")}
for name, f in fields.items():
    k = f["calibration"]
    parts = [f"auroc {k['auroc']:.3f}", f"ece_mass {k['ece_mass']:.3f}"]
    parts += [f"{m} {k[m]:.3f}" for m in extra[f["kind"]]]
    print(f"{name:13} " + ", ".join(parts))
caveat()

queue         auroc 0.960, ece_mass 0.125, brier_multiclass 0.227, nll 0.461
priority      auroc 0.871, ece_mass 0.156, brier_multiclass 0.348, nll 0.570
ticket_type   auroc 0.902, ece_mass 0.141, brier_multiclass 0.306, nll 0.563
frustration   auroc 0.832, ece_mass 0.192, rps 0.099
needs_refund  auroc 0.849, ece_mass 0.096, brier 0.124, log_loss 0.377, auroc_label 0.931
urgent        auroc 0.963, ece_mass 0.126, brier 0.113, log_loss 0.340, auroc_label 0.953

[synthetic placeholder responses: not a measurement of any model; do not cite]


## 5. Accept or escalate

A decision model is usually deployed behind a threshold: accept a decision when its confidence is at
least `tau`, send the rest to a person or a bigger model. `accept_or_escalate` measures that per question,
since confidence scales differ by type (a Noul's never goes below 0.5). Missing answers are always
escalated.

With `target_accuracy`, it also reports the largest share of decisions you could accept while keeping
the accepted ones at that accuracy, and the threshold that gives it.

In [12]:
from stickler.decisions import accept_or_escalate

ops = accept_or_escalate(res, taus=[0.6, 0.7, 0.8, 0.9], target_accuracy=0.95)

print(f"{'question':13} {'AURC':>6} {'coverage at 95%':>16} {'tau':>6}")
for name, f in ops.fields.items():
    tau = "-" if f["tau_at_target"] is None else f"{f['tau_at_target']:.3f}"
    print(f"{name:13} {f['aurc']:>6.3f} {f['coverage_at_target']:>16.2f} {tau:>6}")

print()
print(accept_or_escalate(res, taus=[0.6, 0.7, 0.8, 0.9], target_accuracy=0.95, fields=["queue"]).table())
caveat()

question        AURC  coverage at 95%    tau
queue          0.031             0.77  0.674
priority       0.087             0.49  0.898
ticket_type    0.076             0.51  0.828
frustration    0.203             0.17  0.897
needs_refund   0.047             0.56  0.891
urgent         0.034             0.76  0.782

queue  (n=98, AURC=0.03)
  tau   coverage  accuracy_accepted  errors_escalated
  0.60  0.85      0.92               14 / 21
  0.70  0.74      0.96               18 / 21
  0.80  0.52      1.00               21 / 21
  0.90  0.27      1.00               21 / 21
  coverage at accuracy >= 0.95: 0.77 (tau = 0.674)

[synthetic placeholder responses: not a measurement of any model; do not cite]


Stickler measures at the thresholds you give it; it does not choose one. A threshold read off the same
run it is reported on is optimistic. The honest pattern: pick `tau` on a dev split, then report that
fixed `tau` on a held-out test split.

In [13]:
res_dev, res_test = evaluate(tickets[:50]), evaluate(tickets[50:])
dev = accept_or_escalate(res_dev, target_accuracy=0.95, fields=["queue"]).fields["queue"]
tau = dev["tau_at_target"]
print(f"dev : coverage {dev['coverage_at_target']:.2f} at accuracy >= 0.95, tau = {tau:.3f}\n")
print("test:")
print(accept_or_escalate(res_test, taus=[tau], fields=["queue"]).table())
caveat()

dev : coverage 0.59 at accuracy >= 0.95, tau = 0.772

test:
queue  (n=49, AURC=0.02)
  tau   coverage  accuracy_accepted  errors_escalated
  0.77  0.61      1.00               9 / 9

[synthetic placeholder responses: not a measurement of any model; do not cite]


## 6. Confidence intervals

One run on 100 tickets is a small sample. `confidence_intervals` bootstraps every decision metric after
the run, resampling tickets rather than individual decisions, because decisions about one ticket are
correlated.

In [14]:
from stickler.decisions import confidence_intervals

ci = confidence_intervals(res, n_boot=1000, seed=0)
print(ci["settings"], "\n")
print(f"{'question':13} {'accuracy':>8}  95% interval     {'auroc':>6}  95% interval")
for name, f in fields.items():
    a, al = f["correctness"]["accuracy"], ci[name]["correctness"]["accuracy"]
    u, ul = f["calibration"]["auroc"], ci[name]["calibration"]["auroc"]
    print(
        f"{name:13} {a:>8.3f}  [{al['low']:.3f}, {al['high']:.3f}]   "
        f"{u:>6.3f}  [{ul['low']:.3f}, {ul['high']:.3f}]"
    )
caveat()

{'n_boot': 1000, 'seed': 0, 'alpha': 0.05, 'unit': 'document'} 

question      accuracy  95% interval      auroc  95% interval
queue            0.786  [0.714, 0.867]    0.960  [0.917, 0.993]
priority         0.710  [0.620, 0.800]    0.871  [0.790, 0.937]
ticket_type      0.722  [0.619, 0.804]    0.902  [0.820, 0.965]
frustration      0.556  [0.455, 0.657]    0.832  [0.750, 0.910]
needs_refund     0.823  [0.740, 0.885]    0.849  [0.755, 0.924]
urgent           0.765  [0.683, 0.837]    0.963  [0.920, 0.993]

[synthetic placeholder responses: not a measurement of any model; do not cite]


## Optional: run Strands Decider live

> ## Warning: read before running
>
> - **Downloads several GB.** The first `serve` fetches the model weights from Hugging Face.
> - **Needs Apple silicon or a GPU** for reasonable speed. It runs on CPU, slowly.
> - **Not run in CI.** The cell below is off by default (`RUN_LIVE = False`), so the notebook's stored
>   outputs never come from a live model.
> - **Keep the server on localhost.** It has no authentication; do not expose its port to other machines.

Install and start the server in a terminal:

```bash
pip install strands-decider
strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8000 --device mps
```

On Apple silicon, `--device mlx` is faster than `mps` but needs the `mlx` extra; `--device cuda` and
`--device cpu` cover other machines. The
[strands-decider README](https://github.com/strands-labs/strands-decider#how-to-start-using-strands-decider)
lists the extras each device needs.

Then set `RUN_LIVE = True` and run the cell. It sends a few tickets to the server with the standard
library only, and scores each answer against gold.

In [15]:
import urllib.request

RUN_LIVE = False
URL = "http://127.0.0.1:8000/v1/systemone"


def ask(ticket):
    body = {"state": ticket["state"], "questions": {q: questions[q] for q in ticket["asked"]}}
    req = urllib.request.Request(
        URL, data=json.dumps(body).encode(), headers={"content-type": "application/json"}, method="POST"
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())


if RUN_LIVE:
    for t in tickets[:3]:
        response = ask(t)
        pred = Ticket.from_json(from_systemone(response))
        result = Ticket(**t["gold"]).compare_with(pred)
        print(f"{t['id']} ({response.get('model')}): overall_score {result['overall_score']:.3f}")
        for q in t["asked"]:
            print(f"  {q:13} gold {str(t['gold'][q]):16} predicted {getattr(pred, q)}")
else:
    print("RUN_LIVE is False: skipped. Start the server, set RUN_LIVE = True, and re-run this cell.")

RUN_LIVE is False: skipped. Start the server, set RUN_LIVE = True, and re-run this cell.


### Re-recording the fixture

To replace the synthetic placeholders with real answers for all 100 tickets, run this with the server up:

```bash
cd fixtures/systemone && python record_systemone.py
```

It writes `responses.jsonl` and `provenance.json` only after every ticket has a response, and works with
any System One server (`--url`, `--model`). A recording changes every number above, so re-execute this
notebook in the same commit.